In [12]:
import json
from collections import defaultdict
from pathlib import Path

import pytest
from astropy.coordinates import SkyCoord
from astropy import units as u
from astroquery.alma import Alma

SEARCH_RADIUS_ARCSEC = 60.0
MAX_RESOLUTION_ARCSEC = 1.0
BANDS = [6,7]

In [ ]:
def flatten(obj, prefix=""):
    """Flattens a dictionary or list into a list of key-value pairs"""
    results = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            results.extend(flatten(v, f"{prefix}.{k}" if prefix else k))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            results.extend(flatten(v, f"{prefix}[{i}]"))
    else:
        results.append((prefix, obj))
    return results


def test_output_is_valid_json(agent_output):
    """
    Test that the agent's output.json file is valid JSON (the agent_output fixture will raise an error if the file isn't)
    """
    assert isinstance(agent_output, (dict, list))


def test_reports_a_plausible_observation_count(flat, reference_groups):
    
    

### Fixtures ==> vars

In [ ]:

output_path = Path("test_output.json")
agent_output = json.loads(output_path.read_text())
flat =  list(flatten(agent_output))
reference_coord = SkyCoord.from_name("HH 212")

alma = Alma()
tbl = alma.query_region(
    reference_coord, radius=SEARCH_RADIUS_ARCSEC * u.arcsec,
    band_list=BANDS, public=True, science=True,
)
tbl = tbl[tbl["spatial_resolution"] < MAX_RESOLUTION_ARCSEC]
groups = defaultdict(list)
for row in tbl:
    groups[str(row["member_ous_uid"])].append(row)
reference_groups = groups


### NEW Funcs

In [ ]:
def _find_observation_arrays(obj, path=""):
    """
    Yield (path, list) for every list of dicts in the structure where the
    dicts look like they represent individual observations
    
    NOTE: This function was written using Claude Sonnet 5, with my comments added"""
    
    OBSERVATION_LIKE_KEYS = {"project", "proposal", "band", "resolution", "uid", "mous"}

    if isinstance(obj, dict):
        #If dict, recursively search its values
        for k, v in obj.items():
            yield from _find_observation_arrays(v, f"{path}.{k}" if path else k)
    elif isinstance(obj, list):
        # If list, check if it is a list of dicts...
        if obj and all(isinstance(item, dict) for item in obj):
            sample_keys = " ".join(str(k).lower() for item in obj for k in item.keys())
            # If any of the keys in the dicts match known observation-like keys, yield this list
            if any(term in sample_keys for term in OBSERVATION_LIKE_KEYS):
                yield path, obj
        for i, v in enumerate(obj):
            # Recursively search each item in the list
            yield from _find_observation_arrays(v, f"{path}[{i}]")


def test_observation_count(agent_output, flat, reference_groups):
    expected = len(reference_groups)
    tolerance = max(1, round(0.15 * expected))

    #Level 1:
    #Start by searching for any keys in the output that contain potential observation count identifiers
    potential_identifiers = [
        'obs_count', 'observation_count', 'num_observations',
        'number_of_observations', 'num_obs', 'total_observations',
        'observations_total', 'observations_count', 'obs_total',
        'obs_num', 'obs_number', 'obs_count_total', 'obs_count_num',
        'obs_count_number',
    ]    
    #Check for any keys in the output that contain any of the potential identifiers
    obs_count = [
        value for key, value in flat
        if any(identifier in key.split(".")[-1].split("[")[0].lower() for identifier in potential_identifiers)
    ]
    #Remove any non-integer values from obs_count
    obs_count = [x for x in obs_count if isinstance(x, int) and not isinstance(x, bool)]
    
    if len(obs_count) > 0:
        #If we found any potential observation counts, check if they are within the expected range
        assert any(abs(n - expected) <= tolerance for n in obs_count), (
            f"No reported count near expected {expected} (tolerance {tolerance}) "
            f"found anywhere in the output. Integers present: {sorted(set(obs_count))}"
        )
        return

    # Level 2 (revised): infer count from the structure of an observations-like array
    candidate_arrays = list(_find_observation_arrays(agent_output))
    if candidate_arrays:
        lengths = [len(arr) for _, arr in candidate_arrays]
        assert any(abs(n - expected) <= tolerance for n in lengths), (
            f"No observations-like array with length near {expected} "
            f"(tolerance {tolerance}) found. Array lengths found: {lengths}"
        )
        return

    # If neither level 1 nor level 2 yielded a plausible observation count, fail the test 
    pytest.fail(f"Could not locate any observation count evidence in the output" )

test_observation_count(agent_output, flat, reference_groups)
    

In [ ]:




def test_observation_count(agent_output, flat, reference_groups):
    expected = len(reference_groups)
    tolerance = max(1, round(0.15 * expected))
    print(f"Expected observation count: {expected}")
    print(f"Tolerance: {tolerance}")

    #Level 1:
    #Start by searching for any keys in the output that contain potential observation count identifiers
    potential_identifiers = [
        'obs_count', 'observation_count', 'num_observations',
        'number_of_observations', 'num_obs', 'total_observations',
        'observations_total', 'observations_count', 'obs_total',
        'obs_num', 'obs_number', 'obs_count_total', 'obs_count_num',
        'obs_count_number',
    ]    
    #Check for any keys in the output that contain any of the potential identifiers
    obs_count = [
        value for key, value in flat
        if any(identifier in key.split(".")[-1].split("[")[0].lower() for identifier in potential_identifiers)
    ]
    #Remove any non-integer values from obs_count
    obs_count = [x for x in obs_count if isinstance(x, int) and not isinstance(x, bool)]
    
    if len(obs_count) > 0:
        #If we found any potential observation counts, check if they are within the expected range
        assert any(abs(n - expected) <= tolerance for n in obs_count), (
            f"No reported count near expected {expected} (tolerance {tolerance}) "
            f"found anywhere in the output. Integers present: {sorted(set(obs_count))}"
        )
    else: 
        #Otherwise, search for any integers in the output that are within the tolerance of the expected count
        print("No potential observation count identifiers found in the output. Searching for any integers in the output that are within the tolerance of the expected count.")
        ints_found = [v for _, v in flat if isinstance(v, int) and not isinstance(v, bool)]
        assert any(abs(n - expected) <= tolerance for n in ints_found), (
            f"No reported count near expected {expected} (tolerance {tolerance}) "
            f"found anywhere in the output. Integers present: {sorted(set(ints_found))}"
        )

test_observation_count(agent_output, flat, reference_groups)
    

Expected observation count: 14
Tolerance: 2


In [100]:


print([key.split(".")[-1].split("[")[0].lower() for key, value in flat])

['query_description', 'target', 'ra_deg', 'dec_deg', 'search_radius_arcmin', 'public', 'science', 'bands', 'bands', 'spatial_resolution_arcsec_max', 'num_observations', 'member_ous_uid', 'project_code', 'bands', 'spatial_resolution_arcsec', 'all_spatial_resolutions_arcsec', 'target_names', 's_ra_deg', 's_dec_deg', 'num_spectral_windows', 'member_ous_uid', 'project_code', 'bands', 'spatial_resolution_arcsec', 'all_spatial_resolutions_arcsec', 'target_names', 's_ra_deg', 's_dec_deg', 'num_spectral_windows', 'member_ous_uid', 'project_code', 'bands', 'spatial_resolution_arcsec', 'all_spatial_resolutions_arcsec', 'target_names', 's_ra_deg', 's_dec_deg', 'num_spectral_windows', 'member_ous_uid', 'project_code', 'bands', 'spatial_resolution_arcsec', 'all_spatial_resolutions_arcsec', 'target_names', 's_ra_deg', 's_dec_deg', 'num_spectral_windows', 'member_ous_uid', 'project_code', 'bands', 'spatial_resolution_arcsec', 'all_spatial_resolutions_arcsec', 'target_names', 's_ra_deg', 's_dec_deg', 

In [74]:
output_flat

[('query_description',
  'ALMA archive search for public, science observations near HH 212, Band 6 or 7, spatial_resolution < 1 arcsec'),
 ('target', 'HH 212'),
 ('resolved_coordinates.ra_deg', 85.96420833),
 ('resolved_coordinates.dec_deg', -1.04808333),
 ('search_radius_arcmin', 1.0),
 ('filters.public', True),
 ('filters.science', True),
 ('filters.bands[0]', 6),
 ('filters.bands[1]', 7),
 ('filters.spatial_resolution_arcsec_max', 1.0),
 ('num_observations', 14),
 ('observations[0].member_ous_uid', 'uid://A001/X1273/X1f9'),
 ('observations[0].project_code', '2017.1.00044.S'),
 ('observations[0].bands[0]', 7),
 ('observations[0].spatial_resolution_arcsec', 0.0316734361630697),
 ('observations[0].all_spatial_resolutions_arcsec[0]', 0.0316734361630697),
 ('observations[0].target_names[0]', 'HH_212'),
 ('observations[0].s_ra_deg', 85.96421125000545),
 ('observations[0].s_dec_deg', -1.0481019444447288),
 ('observations[0].num_spectral_windows', 4),
 ('observations[1].member_ous_uid', 'ui

In [75]:
output.keys()

dict_keys(['query_description', 'target', 'resolved_coordinates', 'search_radius_arcmin', 'filters', 'num_observations', 'observations'])